# Classical Machine Learning: Different Ways to Draw a Boundary

Iman Mossavat

Fontys ICT

November 2026

You have already seen a decision tree split a plane into coloured regions, one yes/no question at a time.

Today's question: **what other ways are there?**

| Part | Idea | Model | Time |
|---|---|---|---|
| 0 | One straight line | Logistic regression | 5 min |
| 1 | The widest street | Linear SVM | 15 min |
| 2 | Add a better feature | SVM with kernels | 20 min |
| 3 | Ask your neighbours | k-nearest neighbours (kNN) | 20 min |
| 4 | Teamwork | AdaBoost | 15 min |
| 5 | Which one would you choose? | All of them | 15 min |

### How we work
- **Work in pairs** and swap who types after each part.
- Every part has the same steps: **Predict, Run, Change, Test**.
- Write your prediction down **before** you run a cell.
- Run the cells **from top to bottom**.

## Setup

Run the next cell. It contains helper functions for the plots. You only need to **use** them, not understand them.

- `show(model, X, y)` colours the plane the way the model would.
- `model.score(X, y)` gives the **accuracy**: the fraction of points the model gets right.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier


def show(model, X, y, title="", ax=None, features=None):
    "HELPER: colour the plane according to the model, and draw the data points."
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300)
    x2 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
    xx, yy = np.meshgrid(x1, x2)
    grid = np.c_[xx.ravel(), yy.ravel()]
    if features is not None:
        grid = features(grid)
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=["blue", "red"], alpha=0.2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", s=20, edgecolors="k", linewidths=0.5)
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

---
# Part 0: One straight line

**Logistic regression** gives each input a **weight**, adds them up and turns the sum into a probability. So it can only draw a straight line.

This is our starting point.

In [ ]:
X, y = make_blobs(n_samples=40, centers=[[-2, -2], [2, 2]], cluster_std=1.0, random_state=4)

logistic = LogisticRegression()
logistic.fit(X, y)
show(logistic, X, y, title="Logistic regression")
plt.show()

**Think:** can you draw a *different* straight line that also separates all points correctly? How many such lines are there? How would you pick the best one?

---
# Part 1: The widest street

A **linear Support Vector Machine (SVM)** also draws a straight line. Its rule for choosing the line is:

> Put a **street** between the two groups, make it as **wide** as possible, and drive down the middle.

The points that touch the edges of the street are called **support vectors**.

### Predict
Look at the plot of Part 0. Which points do you think will touch the street? Point at them.

In [ ]:
def show_street(X, y, title="", ax=None):
    "HELPER: train a linear SVM and draw its line, its street and its support vectors."
    # A large C means: do not allow any point inside the street.
    svm = SVC(kernel="linear", C=1000)
    svm.fit(X, y)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    xx, yy = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-5, 5, 300))
    distance = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contour(xx, yy, distance, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", s=30)
    ax.scatter(*svm.support_vectors_.T, s=200, facecolors="none", edgecolors="k", linewidths=2)
    for number, (a, b) in enumerate(X):
        ax.annotate(str(number), (a, b), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_title(title)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    return svm


svm = show_street(X, y, title="Linear SVM: circled points are support vectors")
plt.show()
print("Support vectors are points number:", svm.support_)

### Change
Now we **delete one point** and train again.

1. Choose a point that is **not** circled. Predict: will the line move?
2. Choose a circled point (a support vector). Predict again.

In [ ]:
remove = 0      # <-- change this to the number of the point you want to delete

X_less = np.delete(X, remove, axis=0)
y_less = np.delete(y, remove)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show_street(X, y, title="All points", ax=axes[0])
show_street(X_less, y_less, title=f"Without point {remove} (points are renumbered)", ax=axes[1])
plt.show()

### Test
- Which points decide where the line goes?
- What happens to all the other points?
- Why could a **wide** street be a good idea for points the model has not seen yet?

---
# Part 2: Add a better feature

### Predict
Here is a new dataset: one class forms a circle around the other. Can a straight line separate it?

In [ ]:
X_circles, y_circles = make_circles(n_samples=200, noise=0.08, factor=0.4, random_state=42)

line = SVC(kernel="linear")
line.fit(X_circles, y_circles)
show(line, X_circles, y_circles, title=f"Linear SVM, accuracy {line.score(X_circles, y_circles):.2f}")
plt.show()

We can give the model an extra input: we **calculate a new column** from the old ones.

$$x_3 = x_1^2 + x_2^2$$

**Predict:** what does $x_3$ measure? Which colour will have the larger values?

In [ ]:
def add_feature(X):
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = x1**2 + x2**2          # <-- the new feature. You will change this line.
    return np.c_[x1, x2, x3]    # glue the three columns together


X_new = add_feature(X_circles)
print("Old shape:", X_circles.shape, "  New shape:", X_new.shape)

plt.scatter(X_new[:, 0], X_new[:, 2], c=y_circles, cmap="bwr", s=20, edgecolors="k", linewidths=0.5)
plt.xlabel("x1")
plt.ylabel("x3 (the new feature)")
plt.title("The same points, seen using the new feature")
plt.show()

Can you separate the colours with a straight line now? Let the **same** linear SVM try, using all three columns. Then we colour the original (x1, x2) plane.

In [ ]:
line3 = SVC(kernel="linear")
line3.fit(add_feature(X_circles), y_circles)

accuracy = line3.score(add_feature(X_circles), y_circles)
show(line3, X_circles, y_circles, features=add_feature,
     title=f"Linear SVM with new feature, accuracy {accuracy:.2f}")
plt.show()

The model still uses a flat, straight cut, but in a space with **one extra column**. Seen from the original plane, that cut looks like a circle.

### Change
In `add_feature`, replace the new feature by `x1 * x2`. Run the three cells again.

- Does it work for the circles? Why (not)?
- Can you think of a dataset where `x1 * x2` **would** be the perfect feature? (Remember your answer for Part 5.)

## Kernels: let the SVM do it

Inventing features by hand is hard. A **kernel** lets an SVM behave as if it had many extra features, without you writing them down. The popular **RBF kernel** judges a new point by how **similar** (close) it is to the support vectors.

### Predict
Draw on paper the boundary you expect for the circles and for the moons.

In [ ]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.3, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, X_data, y_data) in zip(axes, [("circles", X_circles, y_circles),
                                             ("moons", X_moons, y_moons)]):
    rbf = SVC(kernel="rbf")      # <-- no add_feature here: only the original x1 and x2
    rbf.fit(X_data, y_data)
    show(rbf, X_data, y_data, ax=ax, title=f"RBF SVM on {name}, accuracy {rbf.score(X_data, y_data):.2f}")
plt.show()

---
# Part 3: Ask your neighbours

**k-nearest neighbours (kNN)** has no weights and no training loop. It simply **remembers all the data**. To classify a new point:

1. Measure the distance from the new point to every known point.
2. Take the **k** closest ones.
3. Let them vote.

### Predict
The star in the next cell is a new point at (0.5, 0.3). Look at the moons plot of Part 2: which colour do you expect its 5 closest neighbours to vote for? Then run the cell.

In [ ]:
new_point = np.array([0.5, 0.3])    # <-- move the star around
k = 5                               # <-- number of neighbours that vote

# 1. distance from the star to every point (Pythagoras)
distances = np.sqrt(((X_moons - new_point) ** 2).sum(axis=1))
# 2. the numbers of the k closest points
closest = np.argsort(distances)[:k]
# 3. the vote
red_votes = y_moons[closest].sum()
blue_votes = k - red_votes

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, title in zip(axes, ["All points", "Zoomed in on the star"]):
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="bwr", s=30, alpha=0.6)
    for i in closest:
        ax.plot([new_point[0], X_moons[i, 0]], [new_point[1], X_moons[i, 1]], "k-", linewidth=1)
    ax.scatter(X_moons[closest, 0], X_moons[closest, 1], s=150, facecolors="none", edgecolors="k")
    ax.scatter(*new_point, marker="*", s=120, c="yellow", edgecolors="k", zorder=3)
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="datalim")

zoom = 1.5 * distances[closest].max()      # window just large enough for the neighbours
axes[1].set_xlim(new_point[0] - zoom, new_point[0] + zoom)
axes[1].set_ylim(new_point[1] - zoom, new_point[1] + zoom)
print(f"Votes: {red_votes} red, {blue_votes} blue")
plt.show()

### Change: the k knob
We now keep many **new points** apart that the model never sees during `fit`. They tell us how well it really does.

**Predict:** what will the boundary look like for `k=1`? And for `k=180`, when we only have 200 points?

In [ ]:
# fresh points from the same source, only used for testing
X_test, y_test = make_moons(n_samples=5000, noise=0.3, random_state=7)

k_values = [1, 15, 180]      # <-- try other values

fig, axes = plt.subplots(1, len(k_values), figsize=(15, 4))
for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_moons, y_moons)
    show(knn, X_moons, y_moons, ax=ax, title=f"k={k}")
    print(f"k={k:3d}   training accuracy {knn.score(X_moons, y_moons):.2f}   "
          f"new-point accuracy {knn.score(X_test, y_test):.2f}")
plt.show()

### Test
- Why is the training accuracy **perfect** for `k=1`? (Who is the closest neighbour of a training point?)
- Which `k` would you use? Which number did you look at to decide?

### Change: stretch one axis
Suppose someone measured `x2` in centimetres instead of metres. All values of `x2` become 100 times bigger. The information is exactly the same.

**Predict:** will kNN give the same accuracy?

In [ ]:
stretch = np.array([1, 100])     # x1 stays, x2 is multiplied by 100

for name, factor in [("original", np.array([1, 1])), ("stretched", stretch)]:
    knn = KNeighborsClassifier(n_neighbors=15)
    knn.fit(X_moons * factor, y_moons)
    print(f"{name:10s} new-point accuracy: {knn.score(X_test * factor, y_test):.2f}")

Explain the difference using the word **distance**. What should you check about your columns before you use kNN?

---
# Part 4: Teamwork

A **decision stump** is a decision tree with only **one question**, for example "is x2 bigger than 0.3?". On its own it is a weak model.

**AdaBoost** builds a team of stumps, one after another:

1. Train a stump.
2. Give **more weight** to the points that are still classified wrongly.
3. Train the next stump on these weighted points.
4. In the end, all stumps vote. Better stumps get a louder voice.

### Predict
What shape can 1 stump draw? And 2 or 3 stumps together?

In [ ]:
team_sizes = [1, 2, 3, 5, 10, 100]      # <-- number of stumps in the team

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, n in zip(axes.ravel(), team_sizes):
    team = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=n, random_state=42)
    team.fit(X_moons, y_moons)
    show(team, X_moons, y_moons, ax=ax, title=f"{n} stumps")
    print(f"{n:3d} stumps   training accuracy {team.score(X_moons, y_moons):.2f}   "
          f"new-point accuracy {team.score(X_test, y_test):.2f}")
plt.tight_layout()
plt.show()

### Test
- Why are all boundaries made of horizontal and vertical pieces?
- Why do 1 and 2 stumps give the same picture? (Think about whose voice is louder.)
- Does a bigger team always do better on **new** points?

---
# Part 5: Which one would you choose?

Here is a **new** dataset. First look at it.

In [ ]:
def make_four_corners(n, seed):
    "HELPER: points in a square, coloured by corner, with a bit of noise."
    random = np.random.default_rng(seed)
    X = random.uniform(-1, 1, size=(n, 2))
    noisy = X + random.normal(0, 0.1, size=(n, 2))
    y = (noisy[:, 0] * noisy[:, 1] > 0).astype(int)
    return X, y


n_train = 200        # <-- later: try 20 and 2000

X_corners, y_corners = make_four_corners(n_train, seed=1)
X_corners_test, y_corners_test = make_four_corners(5000, seed=2)

plt.figure(figsize=(5, 5))
plt.scatter(X_corners[:, 0], X_corners[:, 1], c=y_corners, cmap="bwr", s=20, edgecolors="k", linewidths=0.5)
plt.title("Four corners")
plt.show()

### Predict
Fill in this table **before** you run the next cell (double-click to edit).

| Model | Will it work? (yes / no) | Why? |
|---|---|---|
| Logistic regression | | |
| Linear SVM | | |
| RBF SVM | | |
| kNN (k=15) | | |
| Decision tree (depth 2) | | |
| AdaBoost (100 stumps) | | |

In [ ]:
models = {
    "Logistic regression": LogisticRegression(),
    "Linear SVM": SVC(kernel="linear"),
    "RBF SVM": SVC(kernel="rbf"),
    "kNN (k=15)": KNeighborsClassifier(n_neighbors=15),
    "Decision tree (depth 2)": DecisionTreeClassifier(max_depth=2, random_state=42),
    "AdaBoost (100 stumps)": AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),
                                                n_estimators=100, random_state=42),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, model) in zip(axes.ravel(), models.items()):
    model.fit(X_corners, y_corners)
    score = model.score(X_corners_test, y_corners_test)
    show(model, X_corners, y_corners, ax=ax, title=f"{name}: {score:.2f}")
    print(f"{name:25s} new-point accuracy {score:.2f}")
plt.tight_layout()
plt.show()

### Test
1. Which prediction surprised you most?
2. Make **one** model that failed work well, by giving it **one new feature** (see Part 2).
3. Change `n_train` to 20 and to 2000. Which models need the most data?

In [ ]:
# Your experiment for question 2.
# Hint: write a new function like add_feature, then use it in fit, score and show(..., features=...)

---
# Hand in

Complete the table in your own words (double-click to edit).

| Model | Its idea in one sentence | A shape it draws easily | Where it struggles |
|---|---|---|---|
| Logistic regression | | | |
| SVM | | | |
| kNN | | | |
| AdaBoost | | | |

Then answer: **a customer wants to know why the model made a decision.** Which model would you choose, and why?

### Want to know more?
- Try the models on your own data points, or change the `noise` of the moons.
- scikit-learn [classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html)